# 01 – Data: prostate reports and Gleason labels

Builds the dataset used by every other notebook:

1. Downloads the TCGA-Reports text (Kefeli et al., 2024) and keeps the prostate (PRAD) reports.
2. Downloads the Gleason labels for the same patients from cBioPortal (study `prad_tcga`).
3. Joins reports and labels on the TCGA patient ID.
4. Splits the reports into a small **tuning** set (for developing the prompt and the regex) and an **eval** set (only used for the final runs).
5. Saves the result to `data/prostate_reports.csv`.

In [1]:
import json
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
config = {
    # Sources
    "reports_zip_url": "https://github.com/jkefeli/tcga-path-reports/raw/main/TCGA_Reports.csv.zip",
    "cancer_type_url": "https://raw.githubusercontent.com/jkefeli/tcga-path-reports/main/data/tcga_metadata/tcga_patient_to_cancer_type.csv",
    "cbioportal_url": "https://www.cbioportal.org/api/studies/prad_tcga/clinical-data?clinicalDataType=PATIENT&projection=SUMMARY&pageSize=100000",
    "cancer_type": "PRAD",
    # Paths (relative to the repo root)
    "raw_dir": "data/raw",  # downloads, not committed
    "output_csv": "data/prostate_reports.csv",
    # Split
    "tuning_size": 30,  # number of reports held out for prompt/regex development
    "seed": 13,
}

## Paths and downloads

The notebook lives in `notebooks/`, so paths are resolved from the repo root. Downloads are cached in `data/raw/` and skipped if the file already exists.

In [3]:
# Repo root: the parent folder when the notebook is run from notebooks/, else the current folder
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
raw_dir = ROOT / config["raw_dir"]
raw_dir.mkdir(parents=True, exist_ok=True)


def download(url, path):
    """Download a URL to a local file, unless the file already exists.

    Args:
        url: the URL to fetch.
        path: Path where the file is saved.

    Returns:
        The same path, for convenience.
    """
    if not path.exists():
        print(f"Downloading {url}")
        urllib.request.urlretrieve(url, path)
    return path


reports_zip = download(config["reports_zip_url"], raw_dir / "TCGA_Reports.csv.zip")
cancer_type_csv = download(config["cancer_type_url"], raw_dir / "tcga_patient_to_cancer_type.csv")
clinical_json = download(config["cbioportal_url"], raw_dir / "cbioportal_prad_tcga_patient.json")

## Prostate reports

`TCGA_Reports.csv` has one row per report: `patient_filename` (patient ID + a file UUID) and the OCR `text`. The first 12 characters of the file name are the patient ID, e.g. `TCGA-2A-A8VL`. The cancer type comes from the mapping file in the same GitHub repo.

In [4]:
# The zip also contains a __MACOSX/ entry, so read the CSV by name
with zipfile.ZipFile(reports_zip) as z:
    with z.open("TCGA_Reports.csv") as f:
        reports = pd.read_csv(f)

reports["patient_id"] = reports["patient_filename"].str[:12]
cancer_types = pd.read_csv(cancer_type_csv)

reports = reports.merge(cancer_types, on="patient_id", how="left")
prostate = reports[reports["cancer_type"] == config["cancer_type"]].copy()

print(f"All reports: {len(reports)}")
print(f"Prostate reports: {len(prostate)} ({prostate['patient_id'].nunique()} unique patients)")
assert prostate["patient_id"].is_unique, "Expected one report per patient"

All reports: 9523
Prostate reports: 446 (446 unique patients)


## Gleason labels from cBioPortal

The API returns the clinical data in long format (one row per patient and attribute), so it is pivoted to one row per patient. Only the three Gleason columns are kept. These are **labels only** and must never go into a prompt.

In [5]:
label_columns = {
    "GLEASON_PATTERN_PRIMARY": "label_primary",
    "GLEASON_PATTERN_SECONDARY": "label_secondary",
    "GLEASON_SCORE": "label_score",
}

clinical = pd.DataFrame(json.loads(clinical_json.read_text()))
labels = (
    clinical[clinical["clinicalAttributeId"].isin(label_columns)]
    .pivot(index="patientId", columns="clinicalAttributeId", values="value")
    .rename(columns=label_columns)
    .astype(int)
    .rename_axis("patient_id")
    .reset_index()
)
print(f"Patients with labels: {len(labels)}")

Patients with labels: 500


## Join and sanity checks

In [6]:
data = prostate[["patient_id", "patient_filename", "text"]].merge(labels, on="patient_id", how="left")

# Every report must have a label, and the label must be internally consistent
assert data[list(label_columns.values())].notna().all().all(), "Some reports have no label"
assert (data["label_primary"] + data["label_secondary"] == data["label_score"]).all(), "primary + secondary != score"

print(f"Reports with labels: {len(data)}")
print("\nGleason score distribution:")
print(data["label_score"].value_counts().sort_index().to_string())
print("\nPattern pair distribution:")
print((data["label_primary"].astype(str) + "+" + data["label_secondary"].astype(str)).value_counts().to_string())
print("\nReport length (characters):")
print(data["text"].str.len().describe().round(0).to_string())

Reports with labels: 446

Gleason score distribution:
label_score
6      45
7     237
8      57
9     103
10      4

Pattern pair distribution:
3+4    144
4+3     93
4+5     74
4+4     47
3+3     44
5+4     29
5+3      5
3+5      5
5+5      4
2+4      1

Report length (characters):
count      446.0
mean      3714.0
std       1787.0
min        388.0
25%       2472.0
50%       3451.0
75%       4783.0
max      10928.0


## Tuning / eval split

A small tuning set is held out for developing the prompt and the regex. The eval set is not looked at until the final runs. The split is stratified on the Gleason score so the tuning set has roughly the same class balance, and uses a fixed seed so it is reproducible.

In [7]:
tuning_ids, eval_ids = train_test_split(
    data["patient_id"],
    train_size=config["tuning_size"],
    stratify=data["label_score"],
    random_state=config["seed"],
)
data["split"] = "eval"
data.loc[data["patient_id"].isin(tuning_ids), "split"] = "tuning"

print(pd.crosstab(data["label_score"], data["split"], margins=True).to_string())

split        eval  tuning  All
label_score                   
6              42       3   45
7             221      16  237
8              53       4   57
9              96       7  103
10              4       0    4
All           416      30  446


## Save

In [8]:
output_csv = ROOT / config["output_csv"]
data = data.sort_values(["split", "patient_id"]).reset_index(drop=True)
data.to_csv(output_csv, index=False)
print(f"Saved {len(data)} reports to {output_csv.relative_to(ROOT)}")

Saved 446 reports to data/prostate_reports.csv
